# Khipus.ai
## Introduction to Machine Learning
### Supervised Learning - Multiple Linear Regression with Random Forest Regressor
### Case Study: Car Prices
<span>© Copyright Notice 2025, Khipus.ai - All Rights Reserved.</span>

In [ ]:
# Import necessary packages
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score
import matplotlib.pyplot as plt


## Load and Clean Dataset
Load the dataset and perform basic cleaning, such as removing unnecessary columns and handling missing values.

In [ ]:
# Load the dataset
file_path = 'Automobile_price_data.csv'
df = pd.read_csv(file_path)

# Display basic information and handle any missing values
#df.info()
df = df.dropna()
df.head()

## Feature Selection and Splitting the Data
We will use numerical features for simplicity and split the data into features and target variables.

In [ ]:
# Selecting features and target variable
X = df[['engine-size', 'horsepower', 'city-mpg', 'highway-mpg']]
y = df['price']

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

Multiple Linear Regression:

In this example, engine-size, horsepower, city-mpg, and highway-mpg are the multiple independent variables used to predict the price of the car.


## Training the Model
We will train a Random Forest Regressor on the training data.

In [ ]:
# Import the RandomForestRegressor class from sklearn.ensemble
from sklearn.ensemble import RandomForestRegressor
# Initialize the Random Forest Regressor model with 100 estimators (trees in the forest)
#  and a fixed random state for reproducibility
regressor = RandomForestRegressor(n_estimators=100, random_state=42)

# Train the Random Forest Regressor model using the training data
# X_train contains the feature variables (engine-size, horsepower, city-mpg, highway-mpg)
# y_train contains the target variable (price)
regressor.fit(X_train, y_train)

## Model Evaluation
We will use Mean Squared Error and R^2 Score to evaluate the model's performance.

In [ ]:
# Predict on the test set
y_pred = regressor.predict(X_test)

# Evaluate the model

mse = mean_squared_error(y_test, y_pred) 
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred) 
print(f"Mean Squared Error: {mse}")
print(f"R^2 Score: {r2}")

print(f'Root Mean Squared Error: {rmse}')

Analysis:

R² Score: 0.687

The model explains about 68.7% of the variance in car prices.
Indicates a good level of accuracy in the model's predictions.

Root Mean Squared Error (RMSE): 2361.55

On average, the model's predictions are off by about $2361.55 from the actual car prices.
Easier to interpret as it is in the same units as the target variable (price).

Interpretation:

The model has a good level of accuracy, explaining 68.7% of the variance in car prices.
The average prediction error is about $2361.55, which is relatively moderate.

Next Steps:

Continue improving the model by refining features and trying different models.

Fine-tune the model settings to further reduce errors.

Ensure the data is clean and well-prepared, and handle any outliers or unusual data points.

## Improved code with GitHub Copilot (Prompt: Generate code to improve the resutls)

This improved code adds:

Better feature engineering with power-to-size ratio and fuel efficiency metrics

Proper data scaling using StandardScaler

Hyperparameter tuning using RandomizedSearchCV

Feature importance analysis

More robust missing value handling

In [ ]:
# Import additional required modules
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint

# Handle missing values more carefully
df = df.replace('?', np.nan)  # Replace '?' with NaN
df['price'] = pd.to_numeric(df['price'])
df['horsepower'] = pd.to_numeric(df['horsepower'])
df['engine-size'] = pd.to_numeric(df['engine-size'])
df['city-mpg'] = pd.to_numeric(df['city-mpg'])
df['highway-mpg'] = pd.to_numeric(df['highway-mpg'])

# Feature engineering
df['power_to_size'] = df['horsepower'] / df['engine-size']
df['mpg_ratio'] = df['city-mpg'] / df['highway-mpg']
df['efficiency'] = (df['city-mpg'] + df['highway-mpg']) / 2

# Select features including engineered ones
X = df[[
    'engine-size', 
    'horsepower', 
    'city-mpg', 
    'highway-mpg',
    'power_to_size',
    'mpg_ratio',
    'efficiency'
]]
y = df['price']

# Scale the features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Split the data
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42
)

# Define hyperparameter search space
param_dist = {
    'n_estimators': randint(100, 500),
    'max_depth': [None] + list(range(10, 50, 5)),
    'min_samples_split': randint(2, 20),
    'min_samples_leaf': randint(1, 10),
    'max_features': ['auto', 'sqrt', 'log2']
}

# Initialize Random Forest with RandomizedSearchCV
rf = RandomForestRegressor(random_state=42)
random_search = RandomizedSearchCV(
    rf, 
    param_distributions=param_dist,
    n_iter=100,
    cv=5,
    random_state=42,
    n_jobs=-1
)

# Fit the model
random_search.fit(X_train, y_train)

# Make predictions with best model
y_pred = random_search.predict(X_test)

# Evaluate improved model
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

print(f'Best parameters: {random_search.best_params_}')
print(f'Mean Squared Error: {mse:.2f}')
print(f'Root Mean Squared Error: {rmse:.2f}')
print(f'R² Score: {r2:.4f}')

# Feature importance
feature_importance = pd.DataFrame({
    'feature': X.columns,
    'importance': random_search.best_estimator_.feature_importances_
})
print('\nFeature Importance:')
print(feature_importance.sort_values('importance', ascending=False))
